In [1]:
# from dotenv import load_dotenv
# import os

# load_dotenv()

# api_key = os.getenv("GROQ_API_KEY")
# print(api_key)

In [2]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant"
)

response = llm.invoke("Explain Artificial Intelligence in simple words")

print(response.content)

d:\1stRagProject\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Artificial Intelligence (AI) is a way to make computers and machines think and act like humans. It's like teaching a robot to learn from experience and make decisions on its own.

Imagine you have a friend who can:

1. **Learn**: Your friend can learn from examples and experiences, just like how you learn.
2. **Remember**: Your friend can recall memories and use them to make decisions.
3. **Make decisions**: Your friend can make choices based on what it has learned and remembered.
4. **Improve**: Your friend can get better at making decisions over time.

This is basically what AI does, but instead of a friend, it's a computer program that can do these things. AI uses algorithms (like recipes for computers) to learn from data and make decisions.

There are many types of AI, including:

1. **Machine Learning** (ML): This type of AI learns from data and improves over time.
2. **Natural Language Processing** (NLP): This type of AI helps computers understand and generate human language.
3. 

In [3]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "data/IIT_Foundation-9.pdf"

loader = PyPDFLoader(file_path)
documents = loader.load()

print("Total pages:", len(documents))

Total pages: 537


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

split_documents = text_splitter.split_documents(documents)

print("Total chunks created:", len(split_documents))

Total chunks created: 1555


In [6]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\dinka\AppData\Local\Temp\ipykernel_2916\2671871813.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 454.83it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
from langchain_community.vectorstores import FAISS

# Create FAISS vector store
vectorstore = FAISS.from_documents(split_documents, embeddings)

print("✅ FAISS vector store created successfully!")

✅ FAISS vector store created successfully!


In [8]:
query = "Newton ke laws kya hote hain?"

results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---\n")
    print(doc.page_content[:500])


--- Result 1 ---

Chapter 33.6
Based on the above facts, Galileo concluded that
‘The natural state of a body is not the state of rest. It is the tendency of the body to oppose 
change in its state of motion or rest.’
Newton formulated laws of motion, based on Galileo’s experiment.
NewTON’S FIRST law OF MOTION
‘Every body remains in a state of rest or of uniform motion along a straight line unless and 
until it is compelled by an external force.’
Newton’s first law of motion helps us to understand that
1.  an exte

--- Result 2 ---

with the ball is 0.1s. After striking, if the ball bounces 
back with half its speed before striking, find the force 
which Dhoni exerts on the ball.
 65. T wo groups of students A and B, with two students 
in each group, challenge to accelerate two bodies P 
and Q, respectively placed on a frictionless surface 
as shown in the figure. Which group is successful in 
imparting more acceleration? Justify. Find the accel-
eration of both the bodies.
 66. Laxman

In [9]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
from langchain_groq import ChatGroq
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model="llama-3.1-8b-instant",   #✅ updated working model
    api_key=GROQ_API_KEY
)

query = "Newton ke first law ko simple shabdon me samjhao"

retrieved_docs = vectorstore.similarity_search(query, k=3)
context = "\n\n".join([doc.page_content[:1000] for doc in retrieved_docs])

final_prompt = f"""
Context:
{context}

Question:
{query}

Answer in simple Hindi.
"""

response = llm.invoke(final_prompt)
print(response.content)

Newton ke first law ko simple shabdon me samjhana ho to yeh hai:

"Ek cheez apne mool bhav (natural state) me rahti hai, jo ki thakaan ya samavrti ghoomne ka hissa hai. Isse yeh bhi samajh aata hai ki agar koi bhi cheez apne thakaan ya samavrti ghoomne mein hai to usey koi bhi external bal ka zaroorat hai ki uski thakaan ya samavrti ghoomne mein badal sakti hai.


In [14]:
vectorstore.save_local("faiss_index")